# Lightcast Skills API Data Extraction

This notebook extracts real skills data from the Lightcast Open Skills API for use in the NAB Skill Similarity Engine.

**API Documentation**: https://skills.lightcast.io/
**Scope**: Free access to open skills taxonomy data

We'll start by getting the available versions, then extract the latest skills data.


In [1]:
import requests
import pandas as pd
import json
from datetime import datetime
import time
from pathlib import Path

# API Configuration
CLIENT_ID = "zocd71hodp9qc8wq"
CLIENT_SECRET = "gJ5lJZzL"
SCOPE = "emsi_open"
BASE_URL = "https://emsiservices.com/skills"

print(f"🔗 Lightcast Skills API Configuration")
print(f"Base URL: {BASE_URL}")
print(f"Client ID: {CLIENT_ID}")
print(f"Scope: {SCOPE}")

# Set up data directory
data_dir = Path("../data/skills_library")
data_dir.mkdir(parents=True, exist_ok=True)
print(f"📁 Data directory: {data_dir}")


🔗 Lightcast Skills API Configuration
Base URL: https://emsiservices.com/skills
Client ID: zocd71hodp9qc8wq
Scope: emsi_open
📁 Data directory: ..\data\skills_library


## Step 1: Authentication

Lightcast uses OAuth 2.0 client credentials flow for authentication.


In [2]:
def get_access_token():
    """Get OAuth 2.0 access token from Lightcast"""
    
    auth_url = "https://auth.emsicloud.com/connect/token"
    
    payload = {
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'grant_type': 'client_credentials',
        'scope': SCOPE
    }
    
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded'
    }
    
    print("🔐 Requesting access token...")
    
    try:
        response = requests.post(auth_url, data=payload, headers=headers)
        response.raise_for_status()
        
        token_data = response.json()
        access_token = token_data.get('access_token')
        expires_in = token_data.get('expires_in', 'Unknown')
        
        print(f"✅ Access token obtained successfully")
        print(f"🕒 Expires in: {expires_in} seconds")
        
        return access_token
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Authentication failed: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Response: {e.response.text}")
        return None

# Get authentication token
access_token = get_access_token()
print(f"\nToken available: {access_token is not None}")


🔐 Requesting access token...
✅ Access token obtained successfully
🕒 Expires in: 3600 seconds

Token available: True


## Step 2: Get Available Versions

First, let's fetch the list of available skill taxonomy versions.


In [3]:
def call_api(endpoint, token, params=None):
    """Generic function to call Lightcast API endpoints"""
    
    if not token:
        print("❌ No access token available")
        return None
    
    url = f"{BASE_URL}{endpoint}"
    headers = {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json'
    }
    
    try:
        print(f"🌐 Calling: {url}")
        if params:
            print(f"📋 Params: {params}")
            
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        
        data = response.json()
        print(f"✅ Success: {response.status_code}")
        
        return data
        
    except Exception as e:
        print(f"❌ API call failed: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Status: {e.response.status_code}")
            print(f"Response: {e.response.text}")
        return None

# Get available versions
print("=== FETCHING AVAILABLE VERSIONS ===")
versions_response = call_api("/versions", access_token)

if versions_response:
    print(f"\n📊 Versions Response Structure:")
    print(f"Response type: {type(versions_response)}")
    print(f"Response keys: {list(versions_response.keys()) if isinstance(versions_response, dict) else 'Not a dict'}")
    
    # Display the raw response
    print(f"\n📄 Raw Response:")
    print(json.dumps(versions_response, indent=2))
else:
    print("❌ Failed to get versions")


=== FETCHING AVAILABLE VERSIONS ===
🌐 Calling: https://emsiservices.com/skills/versions
✅ Success: 200

📊 Versions Response Structure:
Response type: <class 'dict'>
Response keys: ['data']

📄 Raw Response:
{
  "data": [
    "9.31",
    "9.30",
    "9.29",
    "9.28",
    "9.27",
    "9.26",
    "9.25",
    "9.24",
    "9.23",
    "9.22",
    "9.21",
    "9.20",
    "9.19",
    "9.18",
    "9.17",
    "9.16",
    "9.15",
    "9.14",
    "9.13",
    "9.12",
    "9.11",
    "9.10",
    "9.9",
    "9.8",
    "9.7",
    "9.6",
    "9.5",
    "9.4",
    "9.3",
    "9.2",
    "9.1",
    "9.0",
    "8.41",
    "8.40",
    "8.39",
    "8.38",
    "8.37",
    "8.36",
    "8.35",
    "8.34",
    "8.33",
    "8.32",
    "8.31",
    "8.30",
    "8.29",
    "8.28",
    "8.27",
    "8.26",
    "8.25",
    "8.24",
    "8.23",
    "8.22",
    "8.21",
    "8.20",
    "8.19",
    "8.18",
    "8.17",
    "8.16",
    "8.15",
    "8.14",
    "8.13",
    "8.12",
    "8.11",
    "8.10",
    "8.9",
    "8.8",


## Step 3: Analyze Versions Response

Once we get the versions response, we'll analyze the structure and extract the version numbers.


In [4]:
# Analyze versions response structure
if versions_response:
    print("=== ANALYZING VERSIONS RESPONSE ===")
    
    # Check if it's a list or dict with data key
    if isinstance(versions_response, list):
        print(f"📋 Direct list of versions: {len(versions_response)} items")
        versions_list = versions_response
    elif isinstance(versions_response, dict) and 'data' in versions_response:
        print(f"📋 Data key found with {len(versions_response['data'])} versions")
        versions_list = versions_response['data']
    else:
        print(f"📋 Unknown structure, trying keys: {list(versions_response.keys())}")
        versions_list = versions_response
    
    # Show first few versions
    if isinstance(versions_list, list) and len(versions_list) > 0:
        print(f"\n🔍 First few versions:")
        for i, version in enumerate(versions_list[:3]):
            print(f"Version {i+1}: {version}")
        
        print(f"\n📊 Total versions available: {len(versions_list)}")
        
        # Extract version numbers if available
        version_numbers = []
        for version in versions_list:
            if isinstance(version, dict):
                # Look for common version fields
                version_num = version.get('version') or version.get('name') or version.get('id')
                if version_num:
                    version_numbers.append(version_num)
        
        if version_numbers:
            print(f"\n🔢 Version numbers found:")
            for vnum in version_numbers[:5]:  # Show first 5
                print(f"  - {vnum}")
            if len(version_numbers) > 5:
                print(f"  ... and {len(version_numbers) - 5} more")
        
        # Save versions data to file
        versions_file = data_dir / "lightcast_versions.json"
        with open(versions_file, 'w') as f:
            json.dump(versions_response, f, indent=2)
        print(f"\n💾 Versions data saved to: {versions_file}")
        
    else:
        print("❌ No version list found in response")
else:
    print("❌ No versions response to analyze")


=== ANALYZING VERSIONS RESPONSE ===
📋 Data key found with 151 versions

🔍 First few versions:
Version 1: 9.31
Version 2: 9.30
Version 3: 9.29

📊 Total versions available: 151

💾 Versions data saved to: ..\data\skills_library\lightcast_versions.json


## Step 4: Extract Skills Data

Now we'll fetch the skills data from the latest version using the skills endpoint.


In [6]:
# Extract ALL versions and fetch comprehensive skills data
if versions_response and 'data' in versions_response:
    all_versions = versions_response['data']  # All 151 versions
    print(f"=== FETCHING COMPREHENSIVE SKILLS DATA FOR ALL {len(all_versions)} VERSIONS ===")
    print(f"📋 Versions: {all_versions[0]} (latest) to {all_versions[-1]} (oldest)")
    
    # Define the fields we want from the API
    skills_fields = "id,name,type,infoUrl,tags,isLanguage,description,descriptionSource,category,subcategory"
    
    # Store all skills data from all versions
    all_skills_data = []
    successful_versions = []
    failed_versions = []
    
    print(f"\n🚀 Starting comprehensive extraction...")
    print(f"⚠️  This will make {len(all_versions)} API calls - please be patient!")
    
    for i, version in enumerate(all_versions, 1):
        print(f"\n🔄 [{i:3d}/{len(all_versions)}] Processing version {version}...")
        
        # Call the skills endpoint for this version
        skills_endpoint = f"/versions/{version}/skills"
        skills_params = {"fields": skills_fields}
        
        skills_response = call_api(skills_endpoint, access_token, params=skills_params)
        
        if skills_response and isinstance(skills_response, dict) and 'data' in skills_response:
            version_skills = skills_response['data']
            print(f"  ✅ Found {len(version_skills):,} skills in version {version}")
            
            # Add version info to each skill
            for skill in version_skills:
                skill['source_version'] = version
                skill['version_order'] = i  # Track processing order (1=newest)
            
            all_skills_data.extend(version_skills)
            successful_versions.append(version)
            
            # Progress summary every 10 versions
            if i % 10 == 0:
                total_skills_so_far = len(all_skills_data)
                print(f"  📊 Progress: {i}/{len(all_versions)} versions, {total_skills_so_far:,} total skills collected")
        
        else:
            print(f"  ❌ Failed to get skills for version {version}")
            failed_versions.append(version)
        
        # Small delay to be respectful to the API
        time.sleep(0.1)
    
    print(f"\n📈 EXTRACTION COMPLETE:")
    print(f"✅ Successful versions: {len(successful_versions)}")
    print(f"❌ Failed versions: {len(failed_versions)}")
    print(f"📊 Total raw skills collected: {len(all_skills_data):,}")
    
    if failed_versions:
        print(f"⚠️  Failed versions: {', '.join(failed_versions[:10])}{'...' if len(failed_versions) > 10 else ''}")
    
    # Show sample of collected data
    if all_skills_data:
        print(f"\n🔍 Sample skill record with version info:")
        sample_skill = all_skills_data[0]
        for key, value in sample_skill.items():
            print(f"  {key}: {value}")
    
else:
    print("❌ No versions available to fetch skills from")


=== FETCHING COMPREHENSIVE SKILLS DATA FOR ALL 151 VERSIONS ===
📋 Versions: 9.31 (latest) to 5.1 (oldest)

🚀 Starting comprehensive extraction...
⚠️  This will make 151 API calls - please be patient!

🔄 [  1/151] Processing version 9.31...
🌐 Calling: https://emsiservices.com/skills/versions/9.31/skills
📋 Params: {'fields': 'id,name,type,infoUrl,tags,isLanguage,description,descriptionSource,category,subcategory'}
✅ Success: 200
  ✅ Found 34,620 skills in version 9.31

🔄 [  2/151] Processing version 9.30...
🌐 Calling: https://emsiservices.com/skills/versions/9.30/skills
📋 Params: {'fields': 'id,name,type,infoUrl,tags,isLanguage,description,descriptionSource,category,subcategory'}
✅ Success: 200
  ✅ Found 34,412 skills in version 9.30

🔄 [  3/151] Processing version 9.29...
🌐 Calling: https://emsiservices.com/skills/versions/9.29/skills
📋 Params: {'fields': 'id,name,type,infoUrl,tags,isLanguage,description,descriptionSource,category,subcategory'}
✅ Success: 200
  ✅ Found 34,382 skills in 

## Step 5: Parse Complex JSON Fields

The skills data contains embedded JSON in several fields that we need to parse properly.


In [7]:
def parse_json_field(field_value):
    """Parse JSON-like fields that might be dicts, lists, or strings"""
    if field_value is None:
        return None, None  # Return (id, name) tuple
    
    if isinstance(field_value, dict):
        # Extract id and name from dict like {'id': 17, 'name': 'Information Technology'}
        field_id = field_value.get('id')
        field_name = field_value.get('name')
        return field_id, field_name
    elif isinstance(field_value, list) and len(field_value) > 0:
        # For lists, take the first item
        first_item = field_value[0]
        if isinstance(first_item, dict):
            return first_item.get('id'), first_item.get('name')
    elif isinstance(field_value, str):
        # Sometimes might be string representation
        try:
            parsed = json.loads(field_value)
            return parse_json_field(parsed)
        except:
            return None, field_value  # Return the string as name
    
    return None, str(field_value) if field_value else None

def process_comprehensive_skills_data(all_skills_data):
    """Process comprehensive skills data and flatten the complex JSON fields"""
    
    processed_skills = []
    
    print(f"🔄 Processing {len(all_skills_data):,} skills from all versions...")
    
    for i, skill in enumerate(all_skills_data):
        # Basic fields (including version tracking)
        processed_skill = {
            'skill_id': skill.get('id'),
            'name': skill.get('name'),
            'description': skill.get('description'),
            'description_source': skill.get('descriptionSource'),
            'info_url': skill.get('infoUrl'),
            'is_language': skill.get('isLanguage', False),
            'source_version': skill.get('source_version'),  # Which version this came from
            'version_order': skill.get('version_order')     # Processing order (1=newest)
        }
        
        # Parse complex JSON fields
        # Category
        category_id, category_name = parse_json_field(skill.get('category'))
        processed_skill['category_id'] = category_id
        processed_skill['category_name'] = category_name
        
        # Subcategory  
        subcategory_id, subcategory_name = parse_json_field(skill.get('subcategory'))
        processed_skill['subcategory_id'] = subcategory_id
        processed_skill['subcategory_name'] = subcategory_name
        
        # Type
        type_id, type_name = parse_json_field(skill.get('type'))
        processed_skill['type_id'] = type_id
        processed_skill['type_name'] = type_name
        
        # Tags (special handling for list)
        tags = skill.get('tags', [])
        if isinstance(tags, list) and len(tags) > 0:
            # Extract all tag names and IDs
            tag_names = []
            tag_ids = []
            for tag in tags:
                if isinstance(tag, dict):
                    if tag.get('name'):
                        tag_names.append(tag.get('name'))
                    if tag.get('id'):
                        tag_ids.append(str(tag.get('id')))
            
            processed_skill['tag_names'] = '|'.join(tag_names) if tag_names else None
            processed_skill['tag_ids'] = '|'.join(tag_ids) if tag_ids else None
        else:
            processed_skill['tag_names'] = None
            processed_skill['tag_ids'] = None
        
        processed_skills.append(processed_skill)
        
        # Progress indicator
        if (i + 1) % 5000 == 0:
            print(f"  ✅ Processed {i + 1:,} skills...")
    
    print(f"✅ Completed processing {len(processed_skills):,} skills")
    return processed_skills

# Process the comprehensive skills data if we have it
if 'all_skills_data' in locals() and all_skills_data:
    processed_skills = process_comprehensive_skills_data(all_skills_data)
    
    # Show sample of processed data
    if processed_skills:
        print(f"\n🔍 Sample processed skill:")
        sample_processed = processed_skills[0]
        for key, value in sample_processed.items():
            print(f"  {key}: {value}")
else:
    print("❌ No comprehensive skills data available to process")


🔄 Processing 2,233,297 skills from all versions...
  ✅ Processed 5,000 skills...
  ✅ Processed 10,000 skills...
  ✅ Processed 15,000 skills...
  ✅ Processed 20,000 skills...
  ✅ Processed 25,000 skills...
  ✅ Processed 30,000 skills...
  ✅ Processed 35,000 skills...
  ✅ Processed 40,000 skills...
  ✅ Processed 45,000 skills...
  ✅ Processed 50,000 skills...
  ✅ Processed 55,000 skills...
  ✅ Processed 60,000 skills...
  ✅ Processed 65,000 skills...
  ✅ Processed 70,000 skills...
  ✅ Processed 75,000 skills...
  ✅ Processed 80,000 skills...
  ✅ Processed 85,000 skills...
  ✅ Processed 90,000 skills...
  ✅ Processed 95,000 skills...
  ✅ Processed 100,000 skills...
  ✅ Processed 105,000 skills...
  ✅ Processed 110,000 skills...
  ✅ Processed 115,000 skills...
  ✅ Processed 120,000 skills...
  ✅ Processed 125,000 skills...
  ✅ Processed 130,000 skills...
  ✅ Processed 135,000 skills...
  ✅ Processed 140,000 skills...
  ✅ Processed 145,000 skills...
  ✅ Processed 150,000 skills...
  ✅ Proce

## Step 6: Deduplicate and Save Comprehensive Skills Data

Remove duplicates, keeping the most recent version of each skill, then save the comprehensive dataset.


In [8]:
# Deduplicate and save comprehensive skills data
if 'processed_skills' in locals() and processed_skills:
    print("=== DEDUPLICATING AND SAVING COMPREHENSIVE SKILLS DATA ===")
    
    # Convert to DataFrame
    raw_skills_df = pd.DataFrame(processed_skills)
    
    print(f"📊 Raw Skills DataFrame Info:")
    print(f"Shape: {raw_skills_df.shape}")
    print(f"Raw skills from all versions: {len(raw_skills_df):,}")
    print(f"Unique skill IDs: {raw_skills_df['skill_id'].nunique():,}")
    
    # Deduplicate: Keep the most recent version of each skill
    # Sort by version_order (1=newest) so latest version is first
    raw_skills_df = raw_skills_df.sort_values('version_order')
    
    # Drop duplicates keeping the first occurrence (which is the most recent)
    skills_df = raw_skills_df.drop_duplicates(subset=['skill_id'], keep='first')
    
    # Add metadata about the deduplication
    skills_df = skills_df.copy()
    skills_df['latest_version'] = skills_df['source_version']  # Rename for clarity
    skills_df = skills_df.drop(['source_version', 'version_order'], axis=1)  # Clean up temp columns
    
    print(f"\n🔄 DEDUPLICATION RESULTS:")
    print(f"Raw skills collected: {len(raw_skills_df):,}")
    print(f"Unique skills after deduplication: {len(skills_df):,}")
    print(f"Duplicates removed: {len(raw_skills_df) - len(skills_df):,}")
    
    # Version analysis
    version_counts = raw_skills_df['source_version'].value_counts().head(10)
    print(f"\n📊 Skills per version (top 10):")
    for version, count in version_counts.items():
        print(f"  v{version}: {count:,} skills")
    
    latest_version_counts = skills_df['latest_version'].value_counts().head(10)
    print(f"\n🆕 Latest appearance per version (top 10):")
    for version, count in latest_version_counts.items():
        print(f"  v{version}: {count:,} unique skills last seen")
    
    # Save to CSV files
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Main comprehensive skills file
    skills_file = data_dir / "lightcast_skills_comprehensive.csv"
    skills_df.to_csv(skills_file, index=False)
    print(f"\n💾 Comprehensive skills data saved to: {skills_file}")
    
    # Also save with timestamp for backup
    backup_file = data_dir / f"lightcast_skills_comprehensive_{timestamp}.csv"
    skills_df.to_csv(backup_file, index=False)
    print(f"📁 Backup saved to: {backup_file}")
    
    # Save raw (pre-deduplication) data for analysis
    raw_file = data_dir / f"lightcast_skills_raw_all_versions_{timestamp}.csv"
    raw_skills_df.to_csv(raw_file, index=False)
    print(f"📄 Raw data (all versions) saved to: {raw_file}")
    
    # Generate comprehensive summary statistics
    print(f"\n📈 COMPREHENSIVE SKILLS LIBRARY STATISTICS:")
    print(f"Total unique skills: {len(skills_df):,}")
    print(f"Categories: {skills_df['category_name'].nunique()}")
    print(f"Subcategories: {skills_df['subcategory_name'].nunique()}")
    print(f"Types: {skills_df['type_name'].nunique()}")
    print(f"Languages: {skills_df['is_language'].sum():,}")
    print(f"With descriptions: {skills_df['description'].notna().sum():,}")
    print(f"Version span: v{skills_df['latest_version'].max()} to v{skills_df['latest_version'].min()}")
    
    # Show category breakdown
    print(f"\n🏷️ Top 10 Categories:")
    category_counts = skills_df['category_name'].value_counts().head(10)
    for cat, count in category_counts.items():
        print(f"  {cat}: {count:,} skills")
    
    # Show sample of final data
    print(f"\n🔍 Sample of comprehensive skills data:")
    sample_cols = ['skill_id', 'name', 'category_name', 'subcategory_name', 'latest_version']
    display(skills_df[sample_cols].head(10))
    
    print(f"\n✅ Comprehensive Lightcast skills extraction complete!")
    print(f"   📊 {len(skills_df):,} unique skills from {len(successful_versions)} versions")
    print(f"   🗂️ Spanning versions {all_versions[0]} to {all_versions[-1]}")
    print(f"   💾 Saved to: {skills_file}")
    
else:
    print("❌ No processed skills data available to save")


=== DEDUPLICATING AND SAVING COMPREHENSIVE SKILLS DATA ===
📊 Raw Skills DataFrame Info:
Shape: (2233297, 16)
Raw skills from all versions: 2,233,297
Unique skill IDs: 38,430

🔄 DEDUPLICATION RESULTS:
Raw skills collected: 2,233,297
Unique skills after deduplication: 38,430
Duplicates removed: 2,194,867

📊 Skills per version (top 10):
  v9.31: 34,620 skills
  v9.30: 34,412 skills
  v9.29: 34,382 skills
  v9.28: 34,350 skills
  v8.8: 34,314 skills
  v8.9: 34,268 skills
  v8.7: 34,244 skills
  v8.14: 34,236 skills
  v8.11: 34,236 skills
  v8.13: 34,229 skills

🆕 Latest appearance per version (top 10):
  v9.31: 34,620 unique skills last seen
  v8.14: 2,025 unique skills last seen
  v8.16: 857 unique skills last seen
  v8.4: 111 unique skills last seen
  v8.9: 102 unique skills last seen
  v8.20: 97 unique skills last seen
  v8.5: 81 unique skills last seen
  v8.8: 67 unique skills last seen
  v8.6: 61 unique skills last seen
  v8.7: 49 unique skills last seen

💾 Comprehensive skills data s

,skill_id,name,category_name,subcategory_name,latest_version
0,KS126XS6CQCFGC3NG79X,.NET Assemblies,Information Technology,Microsoft Development Tools,9.31
23088,ES24CF478CB69A22EDB5,Oral Motor,Health Care,Speech Language Pathology,9.31
23087,ES814687418C1526BA66,Oral Hygiene,Health Care,Oral and Dental Care,9.31
23086,BGS2EBB34153B1BF2321,Oral Health,Health Care,Oral and Dental Care,9.31
23085,KS1277C6FBC93TJWJR5K,Oral Glucose Tolerance Tests,Health Care,General Medical Tests and Procedures,9.31
23084,ES47E9E6CA926AC03419,Oral Food Challenges,Health Care,General Medical Tests and Procedures,9.31
23083,ES3E15C15A1A5B6B7CB5,Oral Expression,Media and Communications,Communication,9.31
23082,ESBDAD3AD0492BB84DC8,Oral Disease Prevention,Health Care,Oral and Dental Care,9.31
23081,ESE8ED62932D770B73EA,Oral Comprehension,Media and Communications,Communication,9.31
23080,ES80955B949B1B0B5623,Oral Care,Health Care,Oral and Dental Care,9.31



✅ Comprehensive Lightcast skills extraction complete!
   📊 38,430 unique skills from 67 versions
   🗂️ Spanning versions 9.31 to 5.1
   💾 Saved to: ..\data\skills_library\lightcast_skills_comprehensive.csv
